# EPB Detector — Colab ramp notebook

Run a full Phase 2-B ramp (e.g. 25 stations × 365 days) on Colab CPU and save the snapshot to your Google Drive.

**Runtime**: this notebook is CPU-bound (pyOASIS uses pandas + scipy). Expect ~50-90 seconds per station-day. A 9 000 station-day ramp takes 8-15 hours; checkpoint progress to Drive.

## Quick start
1. **Runtime ▸ Change runtime type** → CPU (GPU does not help here).
2. Run cells in order. Cell 2 will prompt for Drive permission once.
3. To resume after a Colab disconnect, just run cell 4 again — already-completed station-days are skipped via the manifest.

## What it produces
- `MyDrive/epb-detector/data/processed/features_v2.parquet`
- `MyDrive/epb-detector/data/processed/labels_v2.parquet`
- `MyDrive/epb-detector/data/training_snapshots/v2/{features,labels,splits}.parquet + meta.json + dataset_card.md`
- `MyDrive/epb-detector/cache/manifest.parquet` (resumable)

Pull back to your laptop with `gdown` or `rclone`.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q numpy pandas pyarrow duckdb scipy scikit-learn xgboost matplotlib pyproj georinex astropy pydantic pydantic-settings typer pyyaml shap pandera aacgmv2 rich tabulate scienceplots seaborn

# Install pyOASIS + epb_detector from the project repo. Both packages are
# declared in the same pyproject.toml so a single editable install pulls them
# in. Use a fork by editing REPO_URL below.
REPO_URL = 'https://github.com/tardellirs/plasma-bubble-paper'
BRANCH = 'main'
!rm -rf /content/repo && git clone --depth=1 --branch {BRANCH} {REPO_URL} /content/repo
!pip install -q -e /content/repo

# Sanity check: both packages must resolve before we can ingest.
!cd /content/repo && python -c "import pyOASIS, epb_detector; print('pyOASIS:', pyOASIS.__file__); print('epb_detector:', epb_detector.__file__)"
print('OK')

## 2. Mount Drive (one-time auth) and lay out the workspace

Everything is written under `MyDrive/epb-detector/`. Use a different folder by editing `DRIVE_ROOT`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/epb-detector')
for sub in ('INPUT/RINEX', 'INPUT/ORBITS', 'OUTPUT', 'data/raw', 'data/processed',
            'data/training_snapshots', 'data/space_weather', 'cache', 'models'):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

# Point all epb-detector paths at Drive so a Colab disconnect doesn't lose work.
for k, v in {
    'EPB_PATH_RINEX_INPUT': DRIVE_ROOT / 'INPUT/RINEX',
    'EPB_PATH_ORBIT_INPUT': DRIVE_ROOT / 'INPUT/ORBITS',
    'EPB_PATH_PYOASIS_OUTPUT': DRIVE_ROOT / 'OUTPUT',
    'EPB_PATH_DATA_RAW': DRIVE_ROOT / 'data/raw',
    'EPB_PATH_DATA_PROCESSED': DRIVE_ROOT / 'data/processed',
    'EPB_PATH_DATA_SNAPSHOTS': DRIVE_ROOT / 'data/training_snapshots',
    'EPB_PATH_DATA_SPACE_WEATHER': DRIVE_ROOT / 'data/space_weather',
    'EPB_PATH_CACHE': DRIVE_ROOT / 'cache',
    'EPB_PATH_MODELS': DRIVE_ROOT / 'models',
}.items():
    os.environ[k] = str(v)
    print(k, '→', v)

## 3. Pick the ramp scope

Defaults to a wide Phase 2-B ramp: 8 RBMC stations + the BRAZ/CUIB/MAPA/BELE EIA-crest stations × every 5th day from 2014, 2018, 2020 (solar min) and 2023 (solar max). About 4 000 station-days.

Reduce `STATIONS` or `YEARS_DOY_STEP` if you want a quicker first run.

In [ ]:
STATIONS = ['BOAV', 'MAPA', 'BELE', 'SALU', 'IMPZ', 'BRAZ', 'PALM', 'UFPR', 'POAL']
YEARS_DOY_STEP = {2014: 5, 2018: 7, 2020: 7, 2023: 5, 2024: 5}

from epb_detector.catalog.day_selector import DayKey
from epb_detector.ingest.orchestrator import IngestJob, run_jobs

jobs = []
for year, step in YEARS_DOY_STEP.items():
    last_doy = 366 if year % 4 == 0 else 365
    for doy in range(1, last_doy + 1, step):
        for sta in STATIONS:
            jobs.append(IngestJob(sta, year, doy))
print(f'jobs queued: {len(jobs)}  (≈ {len(jobs) * 60 / 3600:.1f} h sequential)')

## 4. Run ingest (resumable)

Cache lives on Drive — so if Colab disconnects you can re-run this cell to skip already-done station-days.

In [ ]:
import os
os.environ['EPB_INGEST_WORKERS'] = '2'  # Colab has limited cores; 2 is safe
os.environ['MPLBACKEND'] = 'Agg'

recs = run_jobs(jobs)
ok = sum(r.status == 'ok' for r in recs)
fail = sum(r.status == 'failed' for r in recs)
skip = sum(r.status == 'skipped' for r in recs)
print(f'ok={ok}  failed={fail}  skipped={skip}  total={len(recs)}')

## 5. Build features + labels + snapshot v2

In [ ]:
!cd /content/repo && python -m epb_detector.cli features build --version v2
!cd /content/repo && python -m epb_detector.cli labels v2 --features-version v2 --out-version v2
!cd /content/repo && python -m epb_detector.cli dataset snapshot --version v2

## 6. Train + report

In [ ]:
!cd /content/repo && python -m epb_detector.cli train xgb --version v2 --model-id xgb_v0.3.0 --snapshot-id v2

## 7. Pull results back to your laptop

On your laptop, after this notebook has finished:

```bash
# install rclone once: brew install rclone && rclone config  # add a 'gdrive' remote
rclone copy gdrive:epb-detector/data/training_snapshots/v2/ ./data/training_snapshots/v2/
rclone copy gdrive:epb-detector/models/xgb_v0.3.0/ ./models/xgb_v0.3.0/
```

Or just download the parquet files via Drive's web UI — the snapshot folder is small (~10 MB).